# 06 · Dates and strings (Spark)

Dates and strings are where the 3 engines differ the most. Learn the **idea** once, then the names:

| idea | BigQuery | Oracle | Spark |
|---|---|---|---|
| add days | `DATE_ADD(d, INTERVAL 7 DAY)` | `d + 7` | `DATE_ADD(d, 7)` |
| days between | `DATE_DIFF(a, b, DAY)` | `a - b` | `DATEDIFF(a, b)` |
| first day of month | `DATE_TRUNC(d, MONTH)` | `TRUNC(d, 'MM')` | `TRUNC(d, 'MM')` |
| date → text | `FORMAT_DATE('%d/%m/%Y', d)` | `TO_CHAR(d, 'DD/MM/YYYY')` | `DATE_FORMAT(d, 'dd/MM/yyyy')` |
| text → date | `PARSE_DATE('%d/%m/%Y', s)` | `TO_DATE(s, 'DD/MM/YYYY')` | `TO_DATE(s, 'dd/MM/yyyy')` |

**How to run:** Start a cell with `%%sql` to write Spark SQL. In a normal Python cell use the DataFrame API: the tables are ready as variables (`employees`, `orders`, ...). Put your answer in `res` and call `res.show()`.

Tables: see [tables_guide.md](../../jarvis/tables_guide.md). Run the setup cell first (Shift+Enter).

> **Solutions.** Try the practice notebook first: [`spark/practice/06_dates_strings.ipynb`](../../spark/practice/06_dates_strings.ipynb)

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "spark"))
from spark_setup import get_spark, load_tables
from pyspark.sql import functions as F, Window, types as T

spark = get_spark()
t = load_tables(spark)          # also makes SQL views with the same names
employees, departments = t["employees"], t["departments"]
customers, orders = t["customers"], t["orders"]
products, order_items = t["products"], t["order_items"]
user_logins = t["user_logins"]
%load_ext spark_setup

## Q1. Parts of a date  ·  🟢 easy

For each employee show `name`, `hire_year`, `hire_month` (number) and `hire_weekday` (3-letter English day name in CAPITALS, like `MON`).

*Expected result: 30 rows · columns: `name`, `hire_year`, `hire_month`, `hire_weekday`*

**How it works:** `EXTRACT(YEAR/MONTH FROM date)` gives numbers; a format function (`FORMAT_DATE`, `TO_CHAR`, `DATE_FORMAT`) gives the day name as text.

In [ ]:
%%sql
SELECT name,
       EXTRACT(YEAR FROM hire_date)  AS hire_year,
       EXTRACT(MONTH FROM hire_date) AS hire_month,
       UPPER(DATE_FORMAT(hire_date, 'EEE')) AS hire_weekday
FROM employees

,name,hire_year,hire_month,hire_weekday
0,Alice Johnson,2015,3,SUN
1,Bob Smith,2016,6,WED
2,Carol White,2018,1,WED
3,David Brown,2019,4,MON
4,Eva Green,2020,7,WED
5,Frank Moore,2021,9,MON
6,Grace Lee,2022,2,MON
7,Henry Clark,2016,11,SAT
8,Irene Lewis,2017,8,SAT
9,Jack Walker,2018,3,SAT


**DataFrame API:**

In [ ]:
res = employees.select("name",
                       F.year("hire_date").alias("hire_year"),
                       F.month("hire_date").alias("hire_month"),
                       F.upper(F.date_format("hire_date", "EEE")).alias("hire_weekday"))
res.show()

+-------------+---------+----------+------------+
|         name|hire_year|hire_month|hire_weekday|
+-------------+---------+----------+------------+
|Alice Johnson|     2015|         3|         SUN|
|    Bob Smith|     2016|         6|         WED|
|  Carol White|     2018|         1|         WED|
|  David Brown|     2019|         4|         MON|
|    Eva Green|     2020|         7|         WED|
|  Frank Moore|     2021|         9|         MON|
|    Grace Lee|     2022|         2|         MON|
|  Henry Clark|     2016|        11|         SAT|
|  Irene Lewis|     2017|         8|         SAT|
|  Jack Walker|     2018|         3|         SAT|
|   Karen Hall|     2019|        12|         THU|
|    Leo Allen|     2021|         5|         SUN|
|    Mia Young|     2023|         1|         MON|
|  Nathan King|     2017|         2|         TUE|
| Olivia Scott|     2019|         6|         THU|
|   Paul Adams|     2020|        10|         TUE|
|  Quinn Baker|     2024|         3|         MON|


## Q2. Full years of service  ·  🔴 hard

Show `name`, `hire_date` and `years_of_service` = **complete** years from hire_date to 2025-06-30. Someone hired 2016-11-05 has **8** full years on that day, not 9.

*Expected result: 30 rows · columns: `name`, `hire_date`, `years_of_service`*

**How it works:** Count complete months between the dates (minus 1 if the end day is before the start day), then divide by 12 and round **down**.

In [ ]:
%%sql
SELECT name, hire_date,
       FLOOR(MONTHS_BETWEEN(DATE '2025-06-30', hire_date) / 12) AS years_of_service
FROM employees

,name,hire_date,years_of_service
0,Alice Johnson,2015-03-01,10
1,Bob Smith,2016-06-15,9
2,Carol White,2018-01-10,7
3,David Brown,2019-04-22,6
4,Eva Green,2020-07-01,4
5,Frank Moore,2021-09-13,3
6,Grace Lee,2022-02-28,3
7,Henry Clark,2016-11-05,8
8,Irene Lewis,2017-08-19,7
9,Jack Walker,2018-03-03,7


**DataFrame API:**

In [ ]:
res = employees.select("name", "hire_date",
                       F.floor(F.months_between(F.lit("2025-06-30").cast("date"), "hire_date") / 12)
                        .alias("years_of_service"))
res.show()

+-------------+----------+----------------+
|         name| hire_date|years_of_service|
+-------------+----------+----------------+
|Alice Johnson|2015-03-01|              10|
|    Bob Smith|2016-06-15|               9|
|  Carol White|2018-01-10|               7|
|  David Brown|2019-04-22|               6|
|    Eva Green|2020-07-01|               4|
|  Frank Moore|2021-09-13|               3|
|    Grace Lee|2022-02-28|               3|
|  Henry Clark|2016-11-05|               8|
|  Irene Lewis|2017-08-19|               7|
|  Jack Walker|2018-03-03|               7|
|   Karen Hall|2019-12-12|               5|
|    Leo Allen|2021-05-30|               4|
|    Mia Young|2023-01-16|               2|
|  Nathan King|2017-02-14|               8|
| Olivia Scott|2019-06-06|               6|
|   Paul Adams|2020-10-20|               4|
|  Quinn Baker|2024-03-11|               1|
|Rachel Nelson|2016-04-01|               9|
|   Sam Carter|2018-09-09|               6|
|Tina Mitchell|2020-01-27|      

**Why / traps:** Henry Clark: hired 2016-11-05, so on 2025-06-30 he has 8 years and ~8 months: **8** full years. `2025 - 2016 = 9` is wrong.

## Q3. Delivery date and month end  ·  🟢 easy

For orders 1001–1005 (each once) show `order_id`, `order_date`, `due_date` = order_date + 7 days, and `month_end` = last day of the order's month.

*Expected result: 5 rows · columns: `order_id`, `order_date`, `due_date`, `month_end`*

**How it works:** Add 7 days with each engine's date-add; `LAST_DAY(date)` returns the last day of that month.

In [ ]:
%%sql
SELECT DISTINCT order_id, order_date,
       DATE_ADD(order_date, 7) AS due_date,
       LAST_DAY(order_date)    AS month_end
FROM orders
WHERE order_id BETWEEN 1001 AND 1005
ORDER BY order_id

,order_id,order_date,due_date,month_end
0,1001,2023-03-15,2023-03-22,2023-03-31
1,1002,2023-06-13,2023-06-20,2023-06-30
2,1003,2023-06-25,2023-07-02,2023-06-30
3,1004,2023-08-30,2023-09-06,2023-08-31
4,1005,2023-10-17,2023-10-24,2023-10-31


**DataFrame API:**

In [ ]:
res = (orders.filter(F.col("order_id").between(1001, 1005))
       .select("order_id", "order_date",
               F.date_add("order_date", 7).alias("due_date"),
               F.last_day("order_date").alias("month_end"))
       .distinct().orderBy("order_id"))
res.show()

+--------+----------+----------+----------+
|order_id|order_date|  due_date| month_end|
+--------+----------+----------+----------+
|    1001|2023-03-15|2023-03-22|2023-03-31|
|    1002|2023-06-13|2023-06-20|2023-06-30|
|    1003|2023-06-25|2023-07-02|2023-06-30|
|    1004|2023-08-30|2023-09-06|2023-08-31|
|    1005|2023-10-17|2023-10-24|2023-10-31|
+--------+----------+----------+----------+

## Q4. Orders per weekday  ·  🟡 medium

Count different orders per weekday: `weekday` (`MON` … `SUN`) and `num_orders`. Most orders first, then weekday name.

*Expected result: 7 rows · columns: `weekday`, `num_orders`*

**How it works:** Turn each order date into a weekday name, then COUNT DISTINCT order ids per name and sort.

In [ ]:
%%sql
SELECT UPPER(DATE_FORMAT(order_date, 'EEE')) AS weekday,
       COUNT(DISTINCT order_id)              AS num_orders
FROM orders
GROUP BY weekday
ORDER BY num_orders DESC, weekday

,weekday,num_orders
0,WED,19
1,TUE,17
2,MON,15
3,SAT,14
4,SUN,14
5,THU,14
6,FRI,11


**DataFrame API:**

In [ ]:
res = (orders.groupBy(F.upper(F.date_format("order_date", "EEE")).alias("weekday"))
       .agg(F.countDistinct("order_id").alias("num_orders"))
       .orderBy(F.desc("num_orders"), "weekday"))
res.show()

+-------+----------+
|weekday|num_orders|
+-------+----------+
|    WED|        19|
|    TUE|        17|
|    MON|        15|
|    SAT|        14|
|    SUN|        14|
|    THU|        14|
|    FRI|        11|
+-------+----------+

**Why / traps:** Day **numbers** differ: BigQuery `EXTRACT(DAYOFWEEK ...)` is 1 = Sunday; Oracle `TO_CHAR(d, 'D')` depends on the country setting; Spark `DAYOFWEEK` is 1 = Sunday, `WEEKDAY` is 0 = Monday.

## Q5. Recent sign-ups  ·  🟢 easy

Show `customer_id`, `name`, `signup_date` of customers who signed up in the **90 days up to and including** 2024-11-30.

*Expected result: 7 rows · columns: `customer_id`, `name`, `signup_date`*

**How it works:** Compute the start of the window (end date − 90 days) and keep sign-ups after the start and up to the end date.

In [ ]:
%%sql
SELECT customer_id, name, signup_date
FROM customers
WHERE signup_date >  DATE_SUB(DATE '2024-11-30', 90)
  AND signup_date <= DATE '2024-11-30'

,customer_id,name,signup_date
0,4,Maria Kim,2024-11-23
1,10,Sofia Patel,2024-10-27
2,25,Wei Khan,2024-10-05
3,28,Emma Rossi,2024-11-08
4,35,Omar Khan,2024-11-17
5,36,Hannah Muller,2024-11-30
6,37,Ravi Garcia,2024-09-15


**DataFrame API:**

In [ ]:
end = F.lit("2024-11-30").cast("date")
res = (customers.filter((F.col("signup_date") > F.date_sub(end, 90)) & (F.col("signup_date") <= end))
       .select("customer_id", "name", "signup_date"))
res.show()

+-----------+-------------+-----------+
|customer_id|         name|signup_date|
+-----------+-------------+-----------+
|          4|    Maria Kim| 2024-11-23|
|         10|  Sofia Patel| 2024-10-27|
|         25|     Wei Khan| 2024-10-05|
|         28|   Emma Rossi| 2024-11-08|
|         35|    Omar Khan| 2024-11-17|
|         36|Hannah Muller| 2024-11-30|
|         37|  Ravi Garcia| 2024-09-15|
+-----------+-------------+-----------+

**Why / traps:** In real jobs use `CURRENT_DATE` (or `SYSDATE` / `TRUNC(SYSDATE)` in Oracle). A fixed date keeps practice answers stable. Be clear about > vs >= at both ends.

## Q6. First and last name  ·  🟢 easy

Split employee `name` at the space: show `name`, `first_name`, `last_name`.

*Expected result: 30 rows · columns: `name`, `first_name`, `last_name`*

**How it works:** Split the name at the space: array index (BigQuery/Spark) or `SUBSTR` + `INSTR` (position of the space) in Oracle.

In [ ]:
%%sql
SELECT name,
       SPLIT(name, ' ')[0] AS first_name,
       SPLIT(name, ' ')[1] AS last_name
FROM employees

,name,first_name,last_name
0,Alice Johnson,Alice,Johnson
1,Bob Smith,Bob,Smith
2,Carol White,Carol,White
3,David Brown,David,Brown
4,Eva Green,Eva,Green
5,Frank Moore,Frank,Moore
6,Grace Lee,Grace,Lee
7,Henry Clark,Henry,Clark
8,Irene Lewis,Irene,Lewis
9,Jack Walker,Jack,Walker


**DataFrame API:**

In [ ]:
parts = F.split("name", " ")
res = employees.select("name", parts[0].alias("first_name"), parts[1].alias("last_name"))
res.show()

+-------------+----------+---------+
|         name|first_name|last_name|
+-------------+----------+---------+
|Alice Johnson|     Alice|  Johnson|
|    Bob Smith|       Bob|    Smith|
|  Carol White|     Carol|    White|
|  David Brown|     David|    Brown|
|    Eva Green|       Eva|    Green|
|  Frank Moore|     Frank|    Moore|
|    Grace Lee|     Grace|      Lee|
|  Henry Clark|     Henry|    Clark|
|  Irene Lewis|     Irene|    Lewis|
|  Jack Walker|      Jack|   Walker|
|   Karen Hall|     Karen|     Hall|
|    Leo Allen|       Leo|    Allen|
|    Mia Young|       Mia|    Young|
|  Nathan King|    Nathan|     King|
| Olivia Scott|    Olivia|    Scott|
|   Paul Adams|      Paul|    Adams|
|  Quinn Baker|     Quinn|    Baker|
|Rachel Nelson|    Rachel|   Nelson|
|   Sam Carter|       Sam|   Carter|
|Tina Mitchell|      Tina| Mitchell|
+-------------+----------+---------+
only showing top 20 rows

**Why / traps:** `INSTR(s, ' ')` = position of the space. `SUBSTR(s, start, length)`. Positions start at **1** in SQL. BigQuery `OFFSET(0)` and Spark `[0]` start at 0.

Spark also has `SUBSTRING_INDEX(name, ' ', 1)` (text before the 1st space).

## Q7. Initials and lengths  ·  🟢 easy

For customers 1–5 show `name`, `initials` like `A.N.`, `name_upper` and `name_length`.

*Expected result: 5 rows · columns: `name`, `initials`, `name_upper`, `name_length`*

**How it works:** `SUBSTR(text, 1, 1)` = first letter; take the first letter of the first and last name, add dots. `UPPER` and `LENGTH` are the same everywhere.

In [ ]:
%%sql
SELECT name,
       SUBSTR(name, 1, 1) || '.' || SUBSTR(name, INSTR(name, ' ') + 1, 1) || '.' AS initials,
       UPPER(name)  AS name_upper,
       LENGTH(name) AS name_length
FROM customers
WHERE customer_id <= 5
ORDER BY customer_id

,name,initials,name_upper,name_length
0,Aarav Nguyen,A.N.,AARAV NGUYEN,12
1,Priya Silva,P.S.,PRIYA SILVA,11
2,John Lopez,J.L.,JOHN LOPEZ,10
3,Maria Kim,M.K.,MARIA KIM,9
4,Wei Khan,W.K.,WEI KHAN,8


**DataFrame API:**

In [ ]:
res = (customers.filter("customer_id <= 5").orderBy("customer_id")
       .select("name",
               F.concat(F.substring("name", 1, 1), F.lit("."),
                        F.substring(F.split("name", " ")[1], 1, 1), F.lit(".")).alias("initials"),
               F.upper("name").alias("name_upper"),
               F.length("name").alias("name_length")))
res.show()

+------------+--------+------------+-----------+
|        name|initials|  name_upper|name_length|
+------------+--------+------------+-----------+
|Aarav Nguyen|    A.N.|AARAV NGUYEN|         12|
| Priya Silva|    P.S.| PRIYA SILVA|         11|
|  John Lopez|    J.L.|  JOHN LOPEZ|         10|
|   Maria Kim|    M.K.|   MARIA KIM|          9|
|    Wei Khan|    W.K.|    WEI KHAN|          8|
+------------+--------+------------+-----------+

## Q8. Padded employee codes  ·  🟢 easy

Show `emp_id`, `name` and `emp_code` like `E0001`, `E0030` (E + emp_id padded to 4 digits).

*Expected result: 30 rows · columns: `emp_id`, `name`, `emp_code`*

**How it works:** Turn the number into text and pad it on the left with zeros to 4 characters (`LPAD`), then add `E` in front.

In [ ]:
%%sql
SELECT emp_id, name, 'E' || LPAD(CAST(emp_id AS STRING), 4, '0') AS emp_code
FROM employees

,emp_id,name,emp_code
0,1,Alice Johnson,E0001
1,2,Bob Smith,E0002
2,3,Carol White,E0003
3,4,David Brown,E0004
4,5,Eva Green,E0005
5,6,Frank Moore,E0006
6,7,Grace Lee,E0007
7,8,Henry Clark,E0008
8,9,Irene Lewis,E0009
9,10,Jack Walker,E0010


**DataFrame API:**

In [ ]:
res = employees.select("emp_id", "name",
                       F.concat(F.lit("E"), F.lpad(F.col("emp_id").cast("string"), 4, "0")).alias("emp_code"))
res.show()

+------+-------------+--------+
|emp_id|         name|emp_code|
+------+-------------+--------+
|     1|Alice Johnson|   E0001|
|     2|    Bob Smith|   E0002|
|     3|  Carol White|   E0003|
|     4|  David Brown|   E0004|
|     5|    Eva Green|   E0005|
|     6|  Frank Moore|   E0006|
|     7|    Grace Lee|   E0007|
|     8|  Henry Clark|   E0008|
|     9|  Irene Lewis|   E0009|
|    10|  Jack Walker|   E0010|
|    11|   Karen Hall|   E0011|
|    12|    Leo Allen|   E0012|
|    13|    Mia Young|   E0013|
|    14|  Nathan King|   E0014|
|    15| Olivia Scott|   E0015|
|    16|   Paul Adams|   E0016|
|    17|  Quinn Baker|   E0017|
|    18|Rachel Nelson|   E0018|
|    19|   Sam Carter|   E0019|
|    20|Tina Mitchell|   E0020|
+------+-------------+--------+
only showing top 20 rows

**Why / traps:** Fixed-width codes are common in files sent to other systems.

## Q9. Pattern match with a regular expression  ·  🟡 medium

Show `product_name` of products whose name contains a **digit** or the word **Pack**.

*Expected result: 3 rows · columns: `product_name`*

**How it works:** A regular expression `[0-9]|Pack` matches any digit **or** the word Pack. Each engine has its own regex function name.

In [ ]:
%%sql
SELECT product_name
FROM products
WHERE product_name RLIKE '[0-9]|Pack'

,product_name
0,Data Engineering 101
1,Notebook Pack
2,Socks Pack


**DataFrame API:**

In [ ]:
res = products.filter(F.col("product_name").rlike("[0-9]|Pack")).select("product_name")
res.show()

+--------------------+
|        product_name|
+--------------------+
|Data Engineering 101|
|       Notebook Pack|
|          Socks Pack|
+--------------------+

**Why / traps:** Also useful: `REGEXP_REPLACE` (all 3 engines), `REGEXP_EXTRACT` (BigQuery, Spark) / `REGEXP_SUBSTR` (Oracle).

## Q10. Names per department in one line  ·  🟡 medium

Show `dept_id` and `members` = all employee names of the department in one text, A to Z, separated by `, `. Skip NULL dept_id.

*Expected result: 6 rows · columns: `dept_id`, `members`*

**How it works:** Group by department and join all names into one text, sorted A→Z (`STRING_AGG` / `LISTAGG`).

In [ ]:
%%sql
SELECT dept_id, LISTAGG(name, ', ') WITHIN GROUP (ORDER BY name) AS members
FROM employees
WHERE dept_id IS NOT NULL
GROUP BY dept_id
ORDER BY dept_id

,dept_id,members
0,1,"Alice Johnson, Bob Smith, Carol White, David Brown, Eva Green, Frank Moore, Grace Lee"
1,2,"Henry Clark, Irene Lewis, Jack Walker, Karen Hall, Leo Allen, Mia Young"
2,3,"Nathan King, Olivia Scott, Paul Adams, Quinn Baker"
3,4,"Rachel Nelson, Sam Carter, Tina Mitchell, Uma Perez"
4,5,"Victor Roberts, Wendy Turner, Xavier Phillips"
5,6,"Amy Evans, Brian Edwards, Chloe Collins, Yara Campbell, Zane Parker"


**DataFrame API:**

In [ ]:
res = (employees.filter(F.col("dept_id").isNotNull())
       .groupBy("dept_id")
       .agg(F.array_join(F.array_sort(F.collect_list("name")), ", ").alias("members"))
       .orderBy("dept_id"))
res.show()

+-------+--------------------+
|dept_id|             members|
+-------+--------------------+
|      1|Alice Johnson, Bo...|
|      2|Henry Clark, Iren...|
|      3|Nathan King, Oliv...|
|      4|Rachel Nelson, Sa...|
|      5|Victor Roberts, W...|
|      6|Amy Evans, Brian ...|
+-------+--------------------+

**Why / traps:** `LISTAGG` is new in Spark 4. On older Spark use `array_join(array_sort(collect_list(name)), ', ')`.

## Q11. Date to text for a file export  ·  🟢 easy

For orders 1001–1003 (each once) show `order_id` and `order_date_text` in the format `DD/MM/YYYY` (like `15/03/2023`). Files sent to other systems often need this.

*Expected result: 3 rows · columns: `order_id`, `order_date_text`*

**How it works:** Format the date with a pattern: day/month/year. Pattern letters are different in each engine.

In [ ]:
%%sql
SELECT DISTINCT order_id, DATE_FORMAT(order_date, 'dd/MM/yyyy') AS order_date_text
FROM orders
WHERE order_id <= 1003
ORDER BY order_id

,order_id,order_date_text
0,1001,15/03/2023
1,1002,13/06/2023
2,1003,25/06/2023


**DataFrame API:**

In [ ]:
res = (orders.filter("order_id <= 1003")
       .select("order_id", F.date_format("order_date", "dd/MM/yyyy").alias("order_date_text"))
       .distinct().orderBy("order_id"))
res.show()

+--------+---------------+
|order_id|order_date_text|
+--------+---------------+
|    1001|     15/03/2023|
|    1002|     13/06/2023|
|    1003|     25/06/2023|
+--------+---------------+

**Why / traps:** Pattern letters differ! Spark (Java style): `MM` = month, `mm` = **minutes**. Oracle: `MM` = month, `MI` = minutes. BigQuery: `%m` month, `%M` minutes.

## Q12. Text to timestamp  ·  🟢 easy

Turn the text `'2024-03-15 14:30:00'` into a timestamp and show `ts` and `ts_hour` (14). No table needed.

*Expected result: 1 row · columns: `ts`, `ts_hour`*

**How it works:** Parse the text with the matching pattern into a TIMESTAMP, then `EXTRACT(HOUR ...)` / `HOUR(...)`.

In [ ]:
%%sql
SELECT TO_TIMESTAMP('2024-03-15 14:30:00', 'yyyy-MM-dd HH:mm:ss') AS ts,
       HOUR(TO_TIMESTAMP('2024-03-15 14:30:00', 'yyyy-MM-dd HH:mm:ss')) AS ts_hour

,ts,ts_hour
0,2024-03-15 14:30:00,14


**DataFrame API:**

In [ ]:
res = spark.range(1).select(F.to_timestamp(F.lit("2024-03-15 14:30:00"), "yyyy-MM-dd HH:mm:ss").alias("ts")) \
               .withColumn("ts_hour", F.hour("ts"))
res.show()

+-------------------+-------+
|                 ts|ts_hour|
+-------------------+-------+
|2024-03-15 14:30:00|     14|
+-------------------+-------+

## Q13. Logins per ISO week  ·  🟡 medium

Count login **rows** per week. Show `week_start` (the Monday) and `num_logins`, by week.

*Expected result: 9 rows · columns: `week_start`, `num_logins`*

**How it works:** Truncate each date to the Monday of its ISO week, then count rows per week.

In [ ]:
%%sql
SELECT CAST(DATE_TRUNC('WEEK', login_date) AS DATE) AS week_start, COUNT(*) AS num_logins
FROM user_logins
GROUP BY 1
ORDER BY week_start

,week_start,num_logins
0,2024-12-30,9
1,2025-01-06,34
2,2025-01-13,31
3,2025-01-20,30
4,2025-01-27,31
5,2025-02-03,27
6,2025-02-10,32
7,2025-02-17,28
8,2025-02-24,18


**DataFrame API:**

In [ ]:
res = (user_logins.groupBy(F.date_trunc("week", "login_date").cast("date").alias("week_start"))
       .agg(F.count("*").alias("num_logins"))
       .orderBy("week_start"))
res.show()

+----------+----------+
|week_start|num_logins|
+----------+----------+
|2024-12-30|         9|
|2025-01-06|        34|
|2025-01-13|        31|
|2025-01-20|        30|
|2025-01-27|        31|
|2025-02-03|        27|
|2025-02-10|        32|
|2025-02-17|        28|
|2025-02-24|        18|
+----------+----------+

**Why / traps:** ISO weeks start on **Monday**. BigQuery `WEEK` (not ISOWEEK) starts on Sunday.

`DATE_TRUNC` returns a timestamp; CAST it to DATE. `GROUP BY 1` = by the 1st column.